Irish Riftbound ELO Calculator
================================
Runs a pairwise ELO calculation across all Set 1 and Set 2 tournament results.

How it works:
  - Every tournament placement is treated as a round-robin of pairwise results:
    every player who finished above another is considered to have beaten them.
  - K-factor varies by event tier:
      Pre-Rift / Release events  → K = 8
      Locals / Open Play         → K = 16
      Summoner Skirmishes        → K = 40
  - All players start at 900. Players who appeared in Set 1 carry their
    final rating into Set 2.

Usage:
python3 irish_riftbound_elo.py             # print top 50 + key players
python3 irish_riftbound_elo.py --top 100   # print top 100
python3 irish_riftbound_elo.py --all       # print every rated player
python3 irish_riftbound_elo.py --player Nuge  # show a specific player

In [11]:
import argparse
from collections import defaultdict

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

DEFAULT_RATING = 900.0

K_FACTORS = {
    "release": 8,   # Pre-Rift / Release events
    "locals":  16,  # Locals / Open Play
    "ss":      40,  # Summoner Skirmishes
}

# ─────────────────────────────────────────────────────────────────────────────
# ELO ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def lookup(player_id, registry):
    return registry.get(str(player_id), f"Unknown({player_id})")

def expected_score(ra, rb):
    return 1.0 / (1.0 + 10 ** ((rb - ra) / 400.0))

def process_event(standings, tier, ratings):
    """
    Update ratings for one event.
    standings: ordered list of player names, 1st place first.
    Returns total rating points transferred (sum of absolute changes).
    """
    K = K_FACTORS[tier]
    for p in standings:
        if p not in ratings:
            ratings[p] = DEFAULT_RATING

    changes = defaultdict(float)
    pairs = 0
    for i in range(len(standings)):
        for j in range(i + 1, len(standings)):
            winner, loser = standings[i], standings[j]
            e_win = expected_score(ratings[winner], ratings[loser])
            changes[winner] += K * (1.0 - e_win)
            changes[loser]  += K * (0.0 - (1.0 - e_win))
            pairs += 1

    for player, delta in changes.items():
        ratings[player] += delta

    return pairs


# ─────────────────────────────────────────────────────────────────────────────
# EVENT DATA
# Each event is a tuple: (name, date, tier, [standings in finish order])
# tier must be one of: "release", "locals", "ss"
# ─────────────────────────────────────────────────────────────────────────────

SET1_EVENTS = [

    # ── OCTOBER 2025 ──────────────────────────────────────────────────────────
    ("Killarney Release Event",         "Oct 31",  "release", ["Riftbound","swift202020","KevinSpon","Spacemonkey","tanztheman"]),
    #("Skerries Release Event",          "Oct 31",  "release", ["HARANA","Nannerman","KubaKhorne","A Human Skeleton","Kingston","manicMayor","Nuge","Watkin go wrong","Prince Fire Fist","Feralhedgehog","Final Braincell","Topazgoose","Jernney","Aidan Flynn","Guru Ag","PolePole","BaldyPoro","NivalaCross","enzie","Govannan"]),

    # ── NOVEMBER 2025 ─────────────────────────────────────────────────────────
    ("Cork Release Event",              "Nov 1",   "release", ["Deterth","Toasty","Manixous","Ladzy","Akerman","ItsNotOwn","User121565","rob","Local Native","reweed","Twoism","Nitro616","Nemesis","Patch","Crustypants"]),
    #("Dublin City Release Event",       "Nov 1",   "release", ["Forg0","Cia Reeves","Zerleon","Stobie","Prince of Brasov","DaireM","Alfadevil29","batedlawyer884","WeeviltheBrave","bobbypance","User59686","IPenguin","Dovah Juju","thesupajman628","Kealy1994","Kingconway","zeropotential","Sonic Feio","APHELIOS"]),
    #("Tallaght Release Event Nov1",     "Nov 1",   "release", ["Default","Riot Chuchero","Xalarka","Final Braincell","CamoYuki","Skadouchebag","SilentPeruser","Flagcap Zone","Jubx","ScarletDiva","Hutesium","3rknar"]),
    #("Navan Release Event Nov1",        "Nov 1",   "release", ["Blynx","TroupeMaster","Eostrea","lethalrusty","Breener","Down to Clown","TekkaaaaH","bazzilboy","JoeceOfficial"]),
    ("Newtownards Locals Nov1",         "Nov 1",   "locals",  ["Devilminion","Buzsaw Banshee","David Phillips","Rubadubblub","lewcol234","EpithSlayer","Bushrizzlez","Acerbuss","Blake Phillips","crazy","bionicears","Watermelonz","Renoxs"]),
    #("Tallaght Release Event Nov2",     "Nov 2",   "release", ["Zerleon","P0werRangerJaune","Crep","FlexMasterChung","Cia Reeves","BarelyRoss","Whymsy","mattmoons","Honest Utopia","Happyraptor1","Xertacul","DaireM","Sorin The Sad","jazthechicken","This is It","SexyCakeLovesYou","picklericky","McLovett","SilverPhoenix206"]),
    ("Belfast Titanic Release Event",   "Nov 2",   "release", ["CrazyhyperPro","David Phillips","Aldursfc","TheTattooedFox","Blake Phillips","gillynayr","Devilminion","Mccool02","ITopADC","Nonamedrebel99","Cardcoin"]),
    ("Derry Locals Nov3",               "Nov 3",   "locals",  ["Massive4Head","MikeOxverylong69","Banana","Azeromaki","Wozza","DoltWisely","micmule","Hypnos7","Yanna2104","BazookaRooster0","PoonThumper","MothsEXE"]),
    #("Galway Release Event",            "Nov 6",   "release", ["HandsomeRaul","Aynulatac","MollyPaz","Squirle Squad","ADCoPau","Nepiramere","DamjanWolf","Tanakinha","PhantomxxRay","WiltedRose","SacredHeart","DrGLINT25"]),
    #("Skerries Locals Nov6",            "Nov 6",   "locals",  ["Nuge","Topazgoose","KubaKhorne","Lohoris","Nannerman","Final Braincell"]),
    ("Belfast Titanic Locals Nov6",     "Nov 6",   "locals",  ["ITopADC","Acerbuss","David Phillips","DanBoy56","Blake Phillips","Moxi93","Devilminion","Mccool02","CelticMylo","gillynayr"]),
    ("Newtownards Open Play Nov7",      "Nov 7",   "locals",  ["Buzsaw Banshee","Rubadubblub","Sigilita","EpithSlayer","Renoxs"]),
    ("Derry Locals Nov8",               "Nov 8",   "locals",  ["Massive4Head","Banana","micmule","MikeOxverylong69","Wozza","Tetromin0","BazookaRooster0","Wizardoak"]),
    ("Navan Release Event Nov8",        "Nov 8",   "release", ["McK","User156881","Nexodus","Down to Clown","TroupeMaster","User156854"]),
    #("Dublin City Locals Nov8",         "Nov 8",   "locals",  ["Kiwi","Zerleon","Topazgoose","This is It","3rknar","Whymsy","batedlawyer884","ScarletDiva","WeeviltheBrave","Kittenarm","Cia Reeves","Forg0","H3llboyH","Veidos","APHELIOS","Duffsocks","DaireM","Final Braincell","Stobie","SexyCakeLovesYou","Riot Kalandra","Dovah Juju","Kealy1994","BarelyRoss","Prince of Brasov"]),
    ("Tallaght Locals Nov10",           "Nov 10",  "locals",  ["Default","Topazgoose","Zerleon","HARANA","Xalarka","Nannerman","Jernney","Riot Kalandra","Happyraptor1","WeeviltheBrave","JudgePain","CamoYuki","WonderWale"]),
    ("Belfast Titanic Locals Nov13",    "Nov 13",  "locals",  ["LastHuman","gillynayr","Acerbuss","EpithSlayer","Cardcoin","Mccool02","ITopADC","crazy"]),
    ("Limerick Locals Nov13",           "Nov 13",  "locals",  ["ivipivi","AlexBaggs2","Solvorus","PabloY","theunityonline","gwyn"]),
    ("Newtownards Locals Nov14",        "Nov 14",  "locals",  ["Connormcl","Rubadubblub","Aw Balls","Buzsaw Banshee","EpithSlayer","KratosGodofWar","Sigilita","matty"]),
    ("Newry Locals Nov16",              "Nov 16",  "locals",  ["Sciath","NPROWE","Trexos","Seiren","Mark McGuigan","mojokin","IS1090306b2635d8","Conofla"]),
    ("Derry Locals Nov17",              "Nov 17",  "locals",  ["Massive4Head","micmule","MikeOxverylong69","BazookaRooster0","MothsEXE","Banana"]),
    ("Tallaght Locals Nov17",           "Nov 17",  "locals",  ["Whymsy","WeeviltheBrave","batedlawyer884","HARANA","CamoYuki","BarelyRoss","jazthechicken","Andrewfox","Kealy1994","PolePole","Garvin","The Messenger RG","AlfaHorizon","BaldyPoro","Feralhedgehog"]),
    ("Belfast Titanic Locals Nov20",    "Nov 20",  "locals",  ["Acerbuss","Cardcoin","LastHuman","crazy","Reb","PlanetTron"]),
    ("Skerries Locals Nov20",           "Nov 20",  "locals",  ["Final Braincell","CeeJay","Jernney","FBomB11","Topazgoose","KubaKhorne","Nuge","A Human Skeleton","Reggie","Riot Kalandra","Feralhedgehog","Octan behaep","Rando Khan","djsubnet"]),
    ("Galway Locals Nov20",             "Nov 20",  "locals",  ["PhantomxxRay","HandsomeRaul","ADCoPau","Tanakinha","Slee Q"]),
    ("Newtownards Locals Nov21",        "Nov 21",  "locals",  ["Aw Balls","Sigilita","Renoxs","matty"]),
    ("Dublin City Locals Nov22",        "Nov 22",  "locals",  ["Cia Reeves","Topazgoose","Final Braincell","BethanyWinter","seth","jazthechicken","APHELIOS","Dovah Juju","Garvin","BarelyRoss","Andrewfox","Kaze9841","Feralhedgehog","batedlawyer884","F2ng","Forg0","Nexodus","SilverPhoenix206","ObsoleteRuin","DaireM","Honest Utopia","zeropotential"]),
    ("Cork Locals Nov23",               "Nov 23",  "locals",  ["lotuz","Teegz","Ladzy","rob","Patch","reweed","ManuRuP","Damdam","ItsNotOwn","Nemesis","Manixous","Akerman","Local Native","A3ther","Toasty","NebsHD","Crustypants","Arturos","august 14th","shyaboutyou"]),
    ("Newtownards Locals Nov23",        "Nov 23",  "locals",  ["KratosGodofWar","Acerbuss","EpithSlayer","Buzsaw Banshee","Rubadubblub","matty","Renoxs"]),
    ("Newry Locals Nov23",              "Nov 23",  "locals",  ["Sciath","mojokin","Mark McGuigan","ZettaZeptogram"]),
    ("Derry Locals Nov24",              "Nov 24",  "locals",  ["Massive4Head","Banana","micmule"]),
    ("Tallaght Locals Nov24",           "Nov 24",  "locals",  ["Whymsy","WeeviltheBrave","Garvin","BaldyPoro","Nuge","Raizo","HARANA","Feralhedgehog","Nannerman","hraue","The Messenger RG"]),
    ("Belfast Botanic Locals Nov26",    "Nov 26",  "locals",  ["LastHuman","Connormcl","gillynayr","NPROWE","Cardcoin","Seiren","Quinn","XSOverLord","crazy"]),
    ("Belfast Titanic Locals Nov27",    "Nov 27",  "locals",  ["LastHuman","EpithSlayer","crazy","Acerbuss","gillynayr"]),
    ("Skerries Locals Nov27",           "Nov 27",  "locals",  ["Nuge","Garvin","Final Braincell","Nannerman","CeeJay","KubaKhorne","FBomB11","Jernney","HARANA","Prince of Brasov","Feralhedgehog"]),
    ("Navan SS Nov29",                  "Nov 29",  "ss",      ["Whymsy","Facilier","McK","LittleMarMan","Kitsune","Feralhedgehog"]),
    ("Dublin City Locals Nov29",        "Nov 29",  "locals",  ["Garvin","Nuge","F2ng","batedlawyer884","Kiwi","Andrewfox","Kittenarm","Topazgoose","WeeviltheBrave","AlfaHorizon","APHELIOS","Kaze9841","seth","Forg0","Riot Chuchero","jazthechicken","Napita","Kealy1994","SilverPhoenix206","i love numi"]),
    ("Newtownards Locals Nov28",        "Nov 28",  "locals",  ["Rubadubblub","Acerbuss","EpithSlayer","Sigilita","Buzsaw Banshee","Flying Squirrel"]),
    ("Newry Locals Nov30",              "Nov 30",  "locals",  ["Sciath","Mark McGuigan","IS1090306b2635d8","Thyreus123"]),
    ("Newtownards Locals Nov30",        "Nov 30",  "locals",  ["Rubadubblub","KratosGodofWar","EpithSlayer","Buzsaw Banshee"]),

    # ── DECEMBER 2025 ─────────────────────────────────────────────────────────
    ("Tallaght Locals Dec1",            "Dec 1",   "locals",  ["BaldyPoro","Raizo","batedlawyer884","WeeviltheBrave","Nannerman","Kealy1994","This is It","the1gunknights","IrishDante","NMD","hraue","Feralhedgehog"]),
    ("Derry Locals Dec1",               "Dec 1",   "locals",  ["Massive4Head","Banana","micmule","Wozza","Wizardoak"]),
    ("Newtownards Locals Dec5",         "Dec 5",   "locals",  ["Connormcl","Rubadubblub","Buzsaw Banshee","Renoxs"]),
    ("Limerick SS Dec4",                "Dec 4",   "ss",      ["Facilier","ivipivi","kconorfitz","Davidsalazar97","TalkingMaster","gwyn","Hz Spitfyre"]),
    ("Skerries SS Dec4",                "Dec 4",   "ss",      ["Topazgoose","FBomB11","Garvin","CeeJay","Nannerman","Final Braincell","Nuge","batedlawyer884","bob","KubaKhorne","Jernney","HARANA","Kingston","Reggie","User162156","PrinsBrand"]),
    ("Dublin City SS Dec6",             "Dec 6",   "ss",      ["F2ng","Nuge","Kiwi","Whymsy","Forg0","Garvin","APHELIOS","batedlawyer884","PolePole","Kaze9841","HandsomeRaul","BarelyRoss","P0werRangerJaune","seth","Andrewfox","Cannasquawk","DaireM","KubaKhorne","thewalkingdead","BaldyPoro","Final Braincell","Reggie","bob","vandalhandle","Kittenarm","PrN Boizao","hardwiredmagic","This is It","RIOT amypotato","WeeviltheBrave"]),
    ("Derry SS Dec8",                   "Dec 8",   "ss",      ["Massive4Head","BazookaRooster0","micmule"]),
    ("Tallaght Locals Dec8",            "Dec 8",   "locals",  ["WeeviltheBrave","Nannerman","This is It","NMD","PolePole","Raizo","BaldyPoro","the1gunknights","hraue","CamoYuki","The Messenger RG"]),
    ("Galway Locals Dec11",             "Dec 11",  "locals",  ["HandsomeRaul","Aynulatac"]),
    ("Skerries Locals Dec11",           "Dec 11",  "locals",  ["Nuge","KubaKhorne","Final Braincell","Topazgoose","Feralhedgehog","HARANA","Nannerman","FBomB11","Jernney","KING Sybek"]),
    ("Belfast Titanic Locals Dec11",    "Dec 11",  "locals",  ["Connormcl","crazy","Acerbuss","LastHuman","Quinn","Renoxs","Cardcoin","Reb"]),
    ("Dublin City Locals Dec13",        "Dec 13",  "locals",  ["Cia Reeves","F2ng","seth","Forg0","Garvin","batedlawyer884","APHELIOS","Raizo","Reggie","DaireM","SilverPhoenix206"]),
    ("Navan Locals Dec13",              "Dec 13",  "locals",  ["Down to Clown","Breener","bazzilboy","ExtremeLOL109"]),
    ("Tallaght Locals Dec15",           "Dec 15",  "locals",  ["BaldyPoro","PolePole","Raizo","Kealy1994","AlfaHorizon","Feralhedgehog","WeeviltheBrave","LowBeyonder","NMD","hraue","NineBlue","Noobaso","InvestorB","The Messenger RG","the1gunknights"]),
    ("Belfast Titanic Locals Dec18",    "Dec 18",  "locals",  ["LastHuman","EpithSlayer","Sigilita","Cardcoin"]),
    ("Skerries Locals Dec18",           "Dec 18",  "locals",  ["Topazgoose","Nuge","HARANA","Reggie","KubaKhorne","Nannerman","Final Braincell","Grieff","Jernney","KING Sybek","Feralhedgehog"]),
    ("Dublin City Locals Dec20",        "Dec 20",  "locals",  ["Fillabuster","seth","PrN Boizao","Kaze9841","Topazgoose","Kealy1994","Final Braincell","Garvin","LowBeyonder","Forg0","IPenguin","Raizo","BarelyRoss","SilverPhoenix206","djsubnet","Noobaso","BaldyPoro","Cannasquawk","i love numi","Riot Kalandra","Honest Utopia"]),
    ("Cork SS Dec21",                   "Dec 21",  "ss",      ["rob","Local Native","lotuz","Ladzy","A3ther","Teegz","Stephor","Arturos","shyaboutyou","NebsHD","Tammie","TheSpy"]),
    ("Newry SS Dec21",                  "Dec 21",  "ss",      ["Acerbuss","Connormcl","EpithSlayer","Trexos","KING Sybek","Thyreus123"]),
    ("Tallaght Locals Dec22",           "Dec 22",  "locals",  ["BaldyPoro","Nuge","Raizo","AlfaHorizon","Jernney","Feralhedgehog","Topazgoose","EvilOnToast","NMD","The Messenger RG","the1gunknights","Investor B","Gofio Fighter","Raxyel"]),
    ("Dublin City Locals Dec27",        "Dec 27",  "locals",  ["Nuge","Topazgoose","Honest Utopia","Dovah Juju","Kaze9841","Forg0","Garvin","Jernney","i love numi","Reggie","SilverPhoenix206"]),
    ("Newry Locals Dec28",              "Dec 28",  "locals",  ["EpithSlayer","Connormcl","KING Sybek","CelticMylo","Thyreus123"]),
    ("Tallaght Locals Dec29",           "Dec 29",  "locals",  ["Whymsy","WeeviltheBrave","Raizo","LittleMarMan","McK","NMD","Honest Utopia","The Messenger RG","the1gunknights"]),

    # ── JANUARY 2026 ──────────────────────────────────────────────────────────
    ("Dublin City Locals Jan3",         "Jan 3",   "locals",  ["Spardle","Nuge","P0werRangerJaune","Garvin","Reggie","hraue","i love numi","SilverPhoenix206","Dovah Juju","Forg0"]),
    ("Derry Locals Jan5",               "Jan 5",   "locals",  ["Massive4Head","micmule","Reb"]),
    ("Tallaght Locals Jan5",            "Jan 5",   "locals",  ["NMD","Raizo","hraue","Fallwings","WikiDave","The Messenger RG","the1gunknights"]),
    ("Skerries Locals Jan8",            "Jan 8",   "locals",  ["Topazgoose","Nuge","KubaKhorne","Garvin","HARANA","JustOisin","Nannerman","Alex Banana","Jernney","RIFTLAB Jakub","KING Sybek","Reggie","Xenolot"]),
    ("Belfast Titanic SS Jan8",         "Jan 8",   "ss",      ["Sigilita","LastHuman","EpithSlayer","Cardcoin","matty","Reb","PlanetTron"]),
    ("Skerries SS Jan10",               "Jan 10",  "ss",      ["Nuge","HARANA","batedlawyer884","Nannerman","KubaKhorne","Topazgoose","Reggie","Garvin","Dovah Juju","KING Sybek","Jernney","Grieff","Whymsy"]),
    ("Navan Locals Jan10",              "Jan 10",  "locals",  ["Deimne","Down to Clown","bazzilboy","Abel Thbraseike","Breener"]),
    ("Dublin City Locals Jan10",        "Jan 10",  "locals",  ["H3llboyH","IrishDante","RizaHawkseye","SilverPhoenix206","Cannasquawk","Garvin"]),
    ("Tallaght Locals Jan12",           "Jan 12",  "locals",  ["WeeviltheBrave","Garvin","NMD","Feralhedgehog","33fred33","InvestorB","Kealy1994","The Messenger RG","hraue","Ninjacatsif","the1gunknights"]),
    ("Killarney SS Jan14",              "Jan 14",  "ss",      ["tanztheman","Facilier","Airblayde","Riftbound","Spacemonkey"]),
    ("Limerick SS Jan14",               "Jan 14",  "ss",      ["ivipivi","kconorfitz","AlexBaggs2","Nyanshi","HandsomeRaul","PakMan666666"]),
    ("Skerries Locals Jan15",           "Jan 15",  "locals",  ["Topazgoose","FBomB11","Nannerman","Final Braincell","Nuge","RIFTLAB Jakub","CeeJay","KubaKhorne","Xenolot","JustOisin","Garvin","HARANA","Reggie","Feralhedgehog","Alex Banana","KING Sybek","kApe","Technosqueak","jazthechicken"]),
    ("Belfast Titanic Locals Jan15",    "Jan 15",  "locals",  ["EpithSlayer","Reb","PlanetTron","Lion Of Peral","crazy","LindenOfLir","Acerbuss","SunbroSax"]),
    ("Navan SS Jan17",                  "Jan 17",  "ss",      ["Nuge","Topazgoose","Garvin","bazzilboy","Dovah Juju","Down to Clown"]),
    ("Newry Locals Jan18",              "Jan 18",  "locals",  ["Sciath","EpithSlayer","KING Sybek","Thyreus123","LemonArmy","CalmCloud"]),
    ("Tallaght Locals Jan19",           "Jan 19",  "locals",  ["BaldyPoro","Whymsy","WeeviltheBrave","Kealy1994","NMD","InvestorB","Ninjacatsif","Feralhedgehog","Noobaso","The Messenger RG","hraue"]),
    ("Tallaght SS Jan24",               "Jan 24",  "ss",      ["ScarletDiva","Whymsy","BaldyPoro","Spardle","BarelyRoss","Dovah Juju","NMD","Garvin","PolePole","Kealy1994","This is It","Cannasquawk","SilverPhoenix206","Prince of Brasov"]),
    ("Cork SS Jan25",                   "Jan 25",  "ss",      ["Local Native","lotuz","Toasty","Akerman","Ladzy","rob","Patch","A3ther","ClickidyClank","Deterth","Airblayde","Manixous","Akuna05","reweed","Nemesis"]),
    ("Tallaght Locals Jan28",           "Jan 28",  "locals",  ["PolePole","NMD","hraue","The Messenger RG","Guru Ag"]),
    ("Skerries Locals Jan29",           "Jan 29",  "locals",  ["Nuge","CeeJay","Nannerman","Garvin","Cia Reeves","FBomB11","Final Braincell","RIFTLAB Jakub","Jernney","Feralhedgehog","Reggie","HARANA","Xenolot","KubaKhorne","BRone","Dovah Juju","KING Sybek"]),
    ("Dublin City Locals Jan31",        "Jan 31",  "locals",  ["ScarletDiva","Kealy1994","Fillabuster","Garvin","Cia Reeves","Dovah Juju","Ninjacatsif","Noobaso","SilverPhoenix206","BarelyRoss","CrabSoupEnjoyer","vandalhandle","3rknar"]),

    # ── FEBRUARY 2026 ─────────────────────────────────────────────────────────
    ("Cork Locals Feb1",                "Feb 1",   "locals",  ["Toasty","lotuz","Teegz","Local Native","Patch","Akerman","A3ther","Airblayde","Ladzy","shyaboutyou","Reb","ItsNotOwn","Akuna05","whoots","reweed","chad"]),
    ("Tallaght Locals Feb2",            "Feb 2",   "locals",  ["hraue","Raxyel","NMD","InvestorB","Kealy1994","Feralhedgehog","Ninjacatsif","The Messenger RG"]),
    ("Dublin City Locals Feb3",         "Feb 3",   "locals",  ["ScarletDiva","Kealy1994","Spardle","Ninjacatsif","Reggie","GKT SHAIYEN","Kaze9841","SexyCakeLovesYou"]),
]

SET2_EVENTS = [

    # ── FEBRUARY 2026 — PRE-RIFT EVENTS ───────────────────────────────────────
    ("Tallaght Pre-Rift Feb7",          "Feb 7",   "release", ["P0werRangerJaune","AlfaHorizon","WeeviltheBrave","ThatMax11","Soulcatcher","Skapbebag","Default","Riot Chuchero","PolePole","almumuguz","Spardle","JudgePain","NMD","Noobaso","BaldyPoro","the1gunknights","Ninjacatsif","vandalhandle","Raxyel","Kanallero","InvestorB","RIOT amypotato","The Messenger RG","SilverPhoenix206","Happyraptor1"]),
    ("Skerries Pre-Rift Feb7",          "Feb 7",   "release", ["Prince of Brasov","Nuge","F2ng","Reggie","Garvin","GKT SHAIYEN","CrabSoupEnjoyer","Indrik","Cia Reeves","Mooles","Stobie","A Human Skeleton","FBomB11","hraue","Dovah Juju","Feralhedgehog","CeeJay","Kealy1994","excess999d","GKT Blaaden Sama","KING Sybek","Xeratcul","BRone"]),
    ("Derry Pre-Rift Feb7",             "Feb 7",   "release", ["Massive4Head","BazookaRooster0","micmule","DicecreamSlice","RogueLemon","Wizardoak"]),
    ("Newtownards Pre-Rift Feb7",       "Feb 7",   "release", ["Sigilita","k3lso","Buzsaw Banshee","Rubadubblub","Acerbuss","EpithSlayer","Flying Squirrel","XSOverlord"]),
    ("Belfast North Pre-Rift Feb7",     "Feb 7",   "release", ["LastHuman","Thyreus123","Buzsaw Banshee","Rubadubblub","SunbroSax","Crow411","crazy","Acerbuss","PlanetTron","EpithSlayer","Reb","Sigilita","EndirinnKomur","Lion Of Peral","PixleKnight","Cardcoin"]),
    ("Cork Pre-Rift Feb8",              "Feb 8",   "release", ["Lazdy","Crustypants","Teegz","GatchaPlay","Akerman","Patch","rob","ClickidyClank","Manixous","Akuna05","lotuz","NebsHD","Arturos","Niall","Nitro616","reweed","A3ther","shyaboutyou","Local Native","Twoism","Airblayde","whoots"]),
    ("Navan Pre-Rift Feb8",             "Feb 8",   "release", ["Nuge","F2ng","Cia Reeves","Hayan","Blynx","Kealy1994","JoeceOfficial","dickardAdept","Deimne","Stobie","TitanWolfAlpha","TheDongler","TroupeMaster","Bottino"]),
    ("Newry Pre-Rift Feb8",             "Feb 8",   "release", ["Garvin","HARANA","Feralhedgehog","CalmCloud","CelticMylo","LemonArmy","Dovah Juju","Sciath","Gallantmon","KING Sybek","Thyreus123","Reggie","YunaDaisy","NivalaCross","Prince Fire Fist"]),
    ("Belfast North Pre-Rift Feb8",     "Feb 8",   "release", ["LastHuman","Massive4Head","egospiral","EpithSlayer","PlanetTron","Sigilita","micmule","SunbroSax","Devilminion","crazy","Acerbuss","Cardcoin"]),
    ("Killarney Pre-Rift Feb11",        "Feb 11",  "release", ["Airblayde","tanztheman","swift202020","TheSpy","Riftbound","Spacemonkey"]),
    ("Skerries Pre-Rift Feb12",         "Feb 12",  "release", ["Nuge","F2ng","Nannerman","Soulcatcher","HARANA","Cia Reeves","Riot Kalandra","Xenolot","Dovah Juju","Topazgoose"]),
    ("Limerick Pre-Rift Feb12",         "Feb 12",  "release", ["PabloY","ivipivi","Ozzie","FrozenTrickster","theunityonline","prooot","uevyf o","PhantomxxRay","subUWU"]),

    # ── FEBRUARY 2026 — LOCALS ────────────────────────────────────────────────
    ("Newtownards Locals Feb14",        "Feb 14",  "locals",  ["Rubadubblub","Buzsaw Banshee","KappanKimochi","EpithSlayer"]),
    ("Newry Pre-Rift Feb15",            "Feb 15",  "release", ["Sciath","KING Sybek","Thyreus123","YunaDaisy","CelticMylo"]),
    ("Cork Locals Feb15",               "Feb 15",  "locals",  ["Teegz","Crustypants","rob","A3ther","lotuz","Lazdy","Toasty","Manixous","Local Native","reweed","Akerman","Akuna05","Arturos","NebsHD","Tiarnan","Twoism"]),
    ("Tallaght Locals Feb16",           "Feb 16",  "locals",  ["WeeviltheBrave","hraue","DaireM","Nuge","MuffinicK","Kealy1994","The Messenger RG","PolePole","NMD","Ninjacatsif","InvestorB","Noobaso","SFourNine","pumpkinpyee","stroanli","F2ng","BaldyPoro"]),
    ("Derry Locals Feb16",              "Feb 16",  "locals",  ["Massive4Head","BazookaRooster0","BIGFOOT","micmule"]),
    ("Dublin City Locals Feb17",        "Feb 17",  "locals",  ["Topazgoose","ScarletDiva","Cia Reeves","F2ng","Nuge","Kaze9841","Kealy1994","Andrewfox","Noobaso","3rknar","Ninjacatsif","almumuguz","shiraaa","Jester"]),
    ("Killarney Locals Feb18",          "Feb 18",  "locals",  ["tanztheman","Airblayde","swift202020","whoots"]),
    ("Galway Locals Feb19",             "Feb 19",  "locals",  ["Himi91","RincewindEM"]),
    ("Newry Locals Feb21",              "Feb 21",  "locals",  ["Thyreus123","Sciath","SmokeJesus","CelticMylo"]),
    ("Cork Locals Feb22",               "Feb 22",  "locals",  ["Manixous","Lazdy","Patch","reweed","Local Native","Toasty","Arturos","Teegz","rob","A3ther","lotuz","Akerman","Twoism","Airblayde","ItsNotOwn","Crustypants","shyaboutyou","Akuna05","whoots"]),
    ("Tallaght Locals Feb23",           "Feb 23",  "locals",  ["Durag Saiyan","WeeviltheBrave","hraue","IrishDante","Stobie","pumpkinpyee","EvilOnToast","MuffinicK","Ninjacatsif","DaireM","NMD","Kealy1994","Shrapnel Hound","InvestorB","stroanli"]),
    ("Dublin City Locals Feb24",        "Feb 24",  "locals",  ["KubaKhorne","P0werRangerJaune","RIFTLAB Jakub","IrishDante","Kaze9841","Forg0","GKT SHAIYEN","Cannasquawk","Ninjacatsif","Dovah Juju","Kealy1994","Xenolot","BarelyRoss","Durag Saiyan","Reggie","batedlawyer884"]),
    ("Skerries Locals Feb26",           "Feb 26",  "locals",  ["Cia Reeves","RIFTLAB Jakub","Topazgoose","CeeJay","Jernney","Garvin","Nannerman","Dovah Juju","KubaKhorne","Soulcatcher","Xenolot","HARANA","FBomB11","Niall","Viprux","IluvSpagett","Reggie","NivalaCross","KING Sybek","Ezwryy","Prince Fire Fist"]),
    ("Navan Locals Feb28",              "Feb 28",  "locals",  ["bazzilboy","TroupeMaster","Gengar","Muffinman00","AbelThbraseike","Smashanup","Down to Clown"]),

    # ── MARCH 2026 ────────────────────────────────────────────────────────────
    ("Cork Locals Mar1",                "Mar 1",   "locals",  ["lotuz","Teegz","Manixous","Lazdy","rob","A3ther","Crustypants","Toasty","Akuna05","Airblayde","reweed","Local Native","Khalan"]),
    ("Tallaght Locals Mar2",            "Mar 2",   "locals",  ["Nannerman","BaldyPoro","WeeviltheBrave","Feralhedgehog","NMD","Noobaso","Durag Saiyan","Kealy1994","stroanli","Johnica","MuffinicK","RIOT amypotato","Alestrella","The Messenger RG"]),
    ("Derry SS Mar2",                   "Mar 2",   "ss",      ["Massive4Head","micmule","BazookaRooster0","Wizardoak"]),
    ("Dublin City Locals Mar3",         "Mar 3",   "locals",  ["Spardle","Forg0","ScarletDiva","DaireM","KubaKhorne","batedlawyer884","IPenguin","RIFTLAB Jakub","APHELIOS","CrabSoupEnjoyer","Kaze9841","Dovah Juju","MuffinicK","Reggie","Jester","3rknar"]),
    ("Killarney Locals Mar4",           "Mar 4",   "locals",  ["tanztheman","swift202020","Airblayde","Spacemonkey","Riftbound"]),
    ("Skerries Locals Mar5",            "Mar 5",   "locals",  ["Whymsy","Nannerman","Cia Reeves","Niall","Soulcatcher","KubaKhorne","Nuge","Reggie","RIFTLAB Jakub","Xenolot","Jernney","Viprux","HARANA","Alex Banana","FBomB11","Feralhedgehog","Lunastra","BRone","IluvSpagett","Prince of Brasov","Ezwryy"]),
    ("Belfast North SS Mar5",           "Mar 5",   "ss",      ["Rubadubblub","LastHuman","EndirinnKomur","PlanetTron","Buzsaw Banshee","Reb","Sigilita","Crow411","Quinn","Thyreus123","Acerbuss","crazy","Jerrymous3","SunbroSax","Retired Ryze"]),
    ("Dublin City SS Mar7",             "Mar 7",   "ss",      ["Gormfaobhar","PolePole","Kealy1994","P0werRangerJaune","GKT SHAIYEN","batedlawyer884","ScarletDiva","Cannasquawk","Forg0","BarelyRoss","MuffinicK","PipeSupertramp","BaldyPoro","Kaze9841","Ninjacatsif","CourtOfMiracles","RincewindEM","EmberVGC","Whymsy","Himi91","HandsomeRaul","SilverPhoenix206","Aynulatac","Stobie"]),
    ("Skerries SS Mar7",                "Mar 7",   "ss",      ["Nuge","Cia Reeves","RIFTLAB Jakub","KubaKhorne","Reggie","IPenguin","Kingston","Niall","FBomB11","Garvin","Brianmcd48","CeeJay","DaireM","Taleteller","Alex Banana","stroanli","KING Sybek","Feralhedgehog"]),
    ("Navan SS Mar7",                   "Mar 7",   "ss",      ["Pyjamarama1","Deimne","TitanWolfAlpha","Down to Clown","Breener","TroupeMaster","bazzilboy","Abel Thbraseike"]),
    ("Galway Locals Mar8",              "Mar 8",   "locals",  ["dyslexaman","Nepiramere"]),
    ("Cork Locals Mar8",                "Mar 8",   "locals",  ["Akerman","Teegz","Local Native","Crustypants","Airblayde","Deterth","A3ther","Toasty","Patch","Manixous","whoots","rob","Arturos","GatchaPlay","NebsHD","Khalan"]),
    ("Newry SS Mar8",                   "Mar 8",   "ss",      ["mojokin","Thyreus123","Garvin","CalmCloud","EpithSlayer","CelticMylo","LemonArmy","KING Sybek","Ethanjem"]),
]


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_all():
    ratings = {}
    event_count = defaultdict(int)

    for (name, date, tier, standings) in SET1_EVENTS:
        process_event(standings, tier, ratings)
        for p in standings:
            event_count[p] += 1

    set1_final = dict(ratings)

    for (name, date, tier, standings) in SET2_EVENTS:
        process_event(standings, tier, ratings)
        for p in standings:
            event_count[p] += 1

    ranked = sorted(ratings.items(), key=lambda x: -x[1])
    return ranked, ratings, set1_final, event_count


def print_rankings(ranked, ratings, set1_final, event_count, top_n=50, player_filter=None):
    for rank, (pid, rating) in enumerate(ranked, 1):
        name = lookup(pid, registry)
        print(f"{rank:<5} {name:<25} {rating:.1f}")
    if player_filter:
        results = [(p, r) for p, r in ranked if p.lower() == player_filter.lower()]
        if not results:
            # fuzzy match
            results = [(p, r) for p, r in ranked if player_filter.lower() in p.lower()]
        if not results:
            print(f"Player '{player_filter}' not found.")
            return
        rank_lookup = {p: i+1 for i, (p, _) in enumerate(ranked)}
        print(f"\n{'─'*60}")
        for player, rating in results:
            s1 = set1_final.get(player, DEFAULT_RATING)
            n = event_count[player]
            rank = rank_lookup[player]
            diff = rating - s1
            sign = "+" if diff >= 0 else ""
            print(f"  #{rank}  {player}")
            print(f"  Rating : {rating:.1f}")
            print(f"  Set 1  : {s1:.1f}  ({sign}{diff:.1f} across Set 2)")
            print(f"  Events : {n}")
        return

    total = len(ranked)
    ranked = [(p, r) for p, r in ranked if r != set1_final.get(p, DEFAULT_RATING)]
    display = ranked if top_n == 0 else ranked[:top_n]
    rank_lookup = {p: i+1 for i, (p, _) in enumerate(ranked)}

    print(f"\n{'═'*68}")
    print(f"  Irish Riftbound ELO Rankings — March 7th, 2026")
    print(f"  {total} total rated players | {len(SET1_EVENTS)+len(SET2_EVENTS)} events processed")
    print(f"{'═'*68}")
    print(f"  {'Rank':<6}{'Player':<28}{'Rating':<10}{'Set 1':<10}{'Δ':<9}Events")
    print(f"  {'─'*62}")

    for player, rating in display:
        rank = rank_lookup[player]
        s1 = set1_final.get(player, DEFAULT_RATING)
        diff = rating - s1
        sign = "+" if diff >= 0 else ""
        n = event_count[player]
        diff_str = f"{sign}{diff:.0f}"
        print(f"  {rank:<6}{player:<28}{rating:<10.1f}{s1:<10.1f}{diff_str:<9}({n})")

    print(f"{'─'*68}\n")




In [12]:
def main():
    parser = argparse.ArgumentParser(description="Irish Riftbound ELO Calculator")
    parser.add_argument("--top",    type=int, default=50,  help="Number of players to display (default: 50)")
    parser.add_argument("--all",    action="store_true",   help="Show all rated players")
    parser.add_argument("--player", type=str, default=None,help="Look up a specific player by name")
    args = parser.parse_args()

    ranked, ratings, set1_final, event_count = run_all()

    top_n = 0 if args.all else args.top
    print_rankings(ranked, ratings, set1_final, event_count,
                   top_n=top_n, player_filter=args.player)


ranked, ratings, set1_final, event_count = run_all()
print_rankings(ranked, ratings, set1_final, event_count)


════════════════════════════════════════════════════════════════════
  Irish Riftbound ELO Rankings — March 7th, 2026
  346 total rated players | 128 events processed
════════════════════════════════════════════════════════════════════
  Rank  Player                      Rating    Set 1     Δ        Events
  ──────────────────────────────────────────────────────────────
  1     Nuge                        1452.3    1490.7    -38      (25)
  2     Gormfaobhar                 1434.4    900.0     +534     (1)
  3     Cia Reeves                  1383.4    1172.0    +211     (14)
  4     PolePole                    1377.2    1000.8    +376     (10)
  5     Kealy1994                   1358.8    942.4     +416     (21)
  6     Topazgoose                  1316.1    1284.5    +32      (20)
  7     Nannerman                   1304.0    1253.3    +51      (18)
  8     P0werRangerJaune            1283.9    1069.5    +214     (6)
  9     F2ng                        1282.0    1509.1    -227     (9)